In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
import time

import torchvision
from torchvision import datasets, models, transforms

import PIL
import matplotlib.pyplot as plt
import os
import zipfile
from PIL import Image
import pandas as pd
import numpy as np

!pip install split-folders
import splitfolders

import cv2
import dlib
from torchvision.transforms import functional as F

In [ ]:
!pip install deepface

In [ ]:
!pip install efficientnet-pytorch

In [ ]:
!pip install vggface

In [ ]:
!pip install torchvision

In [ ]:
from torchvision.models import mobilenet_v2

In [ ]:
from vggface import VGGFace
from deepface import DeepFace
from efficientnet_pytorch import EfficientNet
from torchvision.models import mobilenet_v2

In [ ]:
from google.colab import drive


In [ ]:
import zipfile


# ? ?
with zipfile.ZipFile(train_zip_path, 'r') as zip_ref:
with zipfile.ZipFile(test_zip_path, 'r') as zip_ref:


In [ ]:
import os
import shutil

#  train ? 

# train ? ? ? 'train' ? ?
for class_name in os.listdir(train_dir):
    class_path = os.path.join(train_dir, class_name)
    if os.path.isdir(class_path) and class_name == 'train':
        shutil.rmtree(class_path)  # 'train' ? ??

# ????? ? ?? ?? ? ?
for class_name in os.listdir(train_dir):
    class_path = os.path.join(train_dir, class_name)
    if os.path.isdir(class_path):
        print(f"Class folder: {class_name}, Files: {os.listdir(class_path)}")


In [ ]:
import os
import shutil
import torch
from torchvision import datasets, transforms
import splitfolders
import cv2
from torch.utils.data import Dataset
from PIL import Image  # PIL ???

# train ??? 8:2?
splitfolders.ratio(train_dir, output=split2_dir, ratio=(.8, .2), group_prefix=None)

# train, val, test ????? ?

# train ? ? ? 'train' ? ?
for class_name in os.listdir(train_data_dir):
    class_path = os.path.join(train_data_dir, class_name)
    if os.path.isdir(class_path) and class_name == 'train':
        shutil.rmtree(class_path)  # 'train' ? ??

for class_name in os.listdir(val_data_dir):
    class_path = os.path.join(val_data_dir, class_name)
    if os.path.isdir(class_path) and class_name == 'train':
        shutil.rmtree(class_path)  # 'train' ? ??

# OpenCV????? ???Dataset ????
class OpenCVDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.classes = os.listdir(data_dir)
        self.image_paths = []

        # ??  ???n        for class_name in self.classes:
            class_dir = os.path.join(data_dir, class_name)
            if os.path.isdir(class_dir):
                for filename in os.listdir(class_dir):
                    if filename.endswith(('.png', '.jpg', '.jpeg')):
                        self.image_paths.append((os.path.join(class_dir, filename), class_name))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path, class_name = self.image_paths[idx]
# External data loading omitted for portfolio archive.
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # BGR??RGB???n        img = Image.fromarray(img)  # NumPy ??PIL ?????n
        if self.transform:
            img = self.transform(img)  # ??????

        # ??????????n        label = self.classes.index(class_name)

        return img, label

# ??????
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(30),
        transforms.RandomAffine(degrees=0, shear=0.1, scale=(0.9, 1.1)),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
        transforms.RandomGrayscale(p=0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# ?? 
train_dataset = OpenCVDataset(train_data_dir, transform=data_transforms['train'])
val_dataset = OpenCVDataset(val_data_dir, transform=data_transforms['val'])
test_dataset = OpenCVDataset(test_data_dir, transform=data_transforms['test'])

# ?????
batch_size = 16
dataloaders = {
    'train': torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=8, pin_memory=True),
    'val': torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=8, pin_memory=True),
    'test': torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=8, pin_memory=True)
}

# ?? ?
dataset_sizes = {'train': len(train_dataset), 'val': len(val_dataset), 'test': len(test_dataset)}

# ????
class_names = train_dataset.classes

# GPU ? ????? device ?
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

#  ?
print(f"Train size: {dataset_sizes['train']}")
print(f"Validation size: {dataset_sizes['val']}")
print(f"Test size: {dataset_sizes['test']}")
print(f"Class names: {class_names}")
print(f"Device: {device}")


In [ ]:
import os
import random
import cv2
import matplotlib.pyplot as plt


# ?? ? (2x5 ?)
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.ravel()  # 2D ??1D???n
# ?????????? ?? 
for idx, class_name in enumerate(class_names[:10]):  # ?10??? 
    # ????? ? ?? ???n    class_dir = os.path.join(train_data_dir, class_name)
    image_files = os.listdir(class_dir)

    # ?? ????? ?
    image_path = os.path.join(class_dir, random.choice(image_files))

    # OpenCV??? ?? ?
# External data loading omitted for portfolio archive.
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # BGR? RGB???n
    # ?????? 
    axes[idx].imshow(img)
    axes[idx].axis('off')  #  ??? ?
    axes[idx].set_title(class_name)  # ???? ?

# ??  
plt.subplots_adjust(wspace=0.3, hspace=0.3)
plt.show()


In [ ]:
# ?, ? ????? ? ?
print(f"Train dataset size: {dataset_sizes['train']}")
print(f"Validation dataset size: {dataset_sizes['val']}")
print(f"Test dataset size: {dataset_sizes['test']}")



---



### **1. ?? ? **

- ResNet50: ?? ??  ?? ?? ? ??.
- VGGFace (ResNet50): VGGFace ????? ResNet50  .
- FaceNet: ? ??????? DeepFace ????.
- EfficientNet: ?? ????? ?? ?? ? ??.
- MobileNetV2: ? ???? ? ???????
- ArcFace: ? ???????  ?? ??? ????

In [ ]:
from torch.cuda.amp import GradScaler, autocast
import matplotlib.pyplot as plt
import torch

# Mixed precision training ?
scaler = GradScaler()

def train_model(model, model_name, loss_function, optimizer, scheduler, max_epochs, dataloaders, dataset_sizes, device):
    file_name = f"{model_name}_model_params.pt"
    best_acc = 0.0
    loss_dict = {"train": [], "val": []}  # ? ?????
    acc_dict = {"train": [], "val": []}   # ????????

    for epoch in range(max_epochs):
        print(f'Epoch {epoch+1}/{max_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()   # ? 
            else:
                model.eval()    # ?? 

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()     # ?? ?0? ?

                with torch.set_grad_enabled(phase == 'train'):
                    with autocast():  # Mixed precision training ???n                        outputs = model(inputs)
                        _, preds = torch.max(outputs, 1)
                        loss = loss_function(outputs, labels)

                    if phase == 'train':
                        scaler.scale(loss).backward()
                        scaler.step(optimizer)
                        scaler.update()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            loss_dict[phase].append(epoch_loss)
            acc_dict[phase].append(epoch_acc.item())

            # ? ?????n            plt.plot(range(len(loss_dict[phase])), loss_dict[phase])
            plt.title(f"{model_name} - {phase} Loss")
            plt.xlabel('Epoch')
            plt.ylabel('Loss')
            plt.savefig(f"{model_name}_{phase}_Loss.png")
            plt.close()

            # ????????n            plt.plot(range(len(acc_dict[phase])), acc_dict[phase])
            plt.title(f"{model_name} - {phase} Accuracy")
            plt.xlabel('Epoch')
            plt.ylabel('Accuracy')
            plt.savefig(f"{model_name}_{phase}_ACC.png")
            plt.close()

            print(f'{phase} Total Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                torch.save(model.state_dict(), file_name)

            if phase == 'val':
                scheduler.step(epoch_loss)   # ?? ?

        print()

    print(f'Best val Acc: {best_acc:.4f}')
    #  ?? ? ? ?/???
    torch.save(loss_dict, f"{model_name}_loss_dict.pth")
    torch.save(acc_dict, f"{model_name}_acc_dict.pth")


**[Mixed precision training]**

https://pytorch.org/docs/stable/amp.html ??n
???? + ? ? ???? ?

**[scheduler ?]**

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=10, verbose=True)

: ?????? ?? ???? ???

-> epoch=10? ?? ?? ????(10%)?? ?


In [ ]:
loss_function = torch.nn.CrossEntropyLoss() # loss function ?
max_epochs = 10 # epoch ???
n_features = len(class_names)

####**1. resnet50**

In [ ]:
model_1 = models.resnet50(pretrained=True)
model_1.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model_1.fc.in_features, n_features)
)
model_1.to(device)

optimizer = torch.optim.Adam(model_1.parameters(), lr=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=10, verbose=True)

train_model(model_1, 'resnet50', loss_function, optimizer, scheduler, max_epochs, dataloaders, dataset_sizes, device)

####**2. EfficientNet**

In [ ]:
model_2 = EfficientNet.from_pretrained('efficientnet-b0').to(device)
model_2._fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model_2._fc.in_features, n_features)
)
model_2.to(device)

optimizer_2 = torch.optim.Adam(model_2.parameters(), lr=1e-5)
scheduler_2 = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_2, mode='min', factor=0.1, patience=10, verbose=True)

train_model(model_2, 'efficientnet', loss_function, optimizer_2, scheduler_2, max_epochs, dataloaders, dataset_sizes, device)

####**3. MobileNetV2**

In [ ]:
from torchvision import models

model_3 = models.mobilenet_v2(pretrained=True).to(device)
model_3.classifier[1] = nn.Linear(model_3.classifier[1].in_features, n_features)
model_3.to(device)

optimizer_3 = torch.optim.Adam(model_3.parameters(), lr=1e-5)
scheduler_3 = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_3, mode='min', factor=0.1, patience=10, verbose=True)

train_model(model_3, 'mobilenetv2', loss_function, optimizer_3, scheduler_3, max_epochs, dataloaders, dataset_sizes, device)

####**4. InceptionV3**

In [ ]:
def inception_train_model(model, model_name, loss_function, optimizer, scheduler, max_epochs, dataloaders, dataset_sizes, device):
    best_model_wts = model.state_dict()
    best_acc = 0.0
    for epoch in range(max_epochs):
        print(f'Epoch {epoch + 1}/{max_epochs}')
        print('-' * 10)

        # ?epoch ???? ?
        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            running_loss = 0.0
            running_corrects = 0

            # ???
            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)

                #  ??n                optimizer.zero_grad()

                with autocast():  # Mixed precision training ???n                    outputs = model(inputs)
                    # outputs[0]? logits???
                    logits = outputs[0] if isinstance(outputs, tuple) else outputs  # outputs ???? [0]???
                    _, preds = torch.max(logits, 1)  # logits???? ??
                    loss = loss_function(logits, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # ?????  ???n            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = model.state_dict()

        # ???? ??
        scheduler.step(epoch_loss)

    print(f'Best val Acc: {best_acc:.4f}')
    model.load_state_dict(best_model_wts)
    return model

In [ ]:
torch.cuda.empty_cache()

In [ ]:
model_4 = models.inception_v3(pretrained=True)
model_4.AuxLogits.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model_4.AuxLogits.fc.in_features, n_features)
)
model_4.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model_4.fc.in_features, n_features)
)
model_4.to(device)

inception_transform = transforms.Compose([
    transforms.Resize((299, 299)),  # InceptionV3??? ?
    transforms.ToTensor(),          # ????n    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ???n])

inception_train_dataset = OpenCVDataset(train_data_dir, transform=inception_transform)  # InceptionV3?? transform ?
inception_val_dataset = OpenCVDataset(val_data_dir, transform=inception_transform)      # InceptionV3?? transform ?
inception_test_dataset = OpenCVDataset(test_data_dir, transform=inception_transform)    # InceptionV3?? transform ?

# ?????
batch_size = 16
inception_dataloaders = {
    'train': torch.utils.data.DataLoader(inception_train_dataset, batch_size=batch_size, shuffle=True, num_workers=8, pin_memory=True),
    'val': torch.utils.data.DataLoader(inception_val_dataset, batch_size=batch_size, shuffle=False, num_workers=8, pin_memory=True),
    'test': torch.utils.data.DataLoader(inception_test_dataset, batch_size=batch_size, shuffle=False, num_workers=8, pin_memory=True)
}

# ??? ?? ?
optimizer_4 = torch.optim.Adam(model_4.parameters(), lr=1e-5)
scheduler_4 = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_4, mode='min', factor=0.1, patience=10, verbose=True)

#  ?
inception_train_model(model_4, 'inception_v3', loss_function, optimizer_4, scheduler_4, max_epochs, inception_dataloaders, dataset_sizes, device)

In [ ]:
import cv2
import torch
from torchvision import models
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.cuda.amp import GradScaler, autocast

# ? ?? (Haar Cascade Classifier ?)
def detect_faces_haar(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
    return faces

# ? ???? ?
def train_with_face_detection_haar(model, model_name, loss_function, optimizer, scheduler, max_epochs, dataloaders, dataset_sizes, device):
    best_acc = 0.0
    loss_dict = {"train": [], "val": []}
    acc_dict = {"train": [], "val": []}

    for epoch in range(max_epochs):
        print(f'Epoch {epoch+1}/{max_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    with autocast():
                        outputs = model(inputs)
                        _, preds = torch.max(outputs, 1)
                        loss = loss_function(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            loss_dict[phase].append(epoch_loss)
            acc_dict[phase].append(epoch_acc.item())

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                torch.save(model.state_dict(), f"{model_name}_best_model.pth")

            if phase == 'val':
                scheduler.step(epoch_loss)

    print(f'Best val Acc: {best_acc:.4f}')
    torch.save(loss_dict, f"{model_name}_loss_dict.pth")
    torch.save(acc_dict, f"{model_name}_acc_dict.pth")
    return model

# ResNet50  ?
model_5 = models.resnet50(pretrained=True)
model_5.fc = nn.Linear(model_5.fc.in_features, n_features)  # ????  ??
model_5.to(device)

optimizer_5 = torch.optim.Adam(model_5.parameters(), lr=1e-5)
scheduler_5 = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_5, mode='min', factor=0.1, patience=10, verbose=True)

train_with_face_detection_haar(model_5, 'resnet50_haar', loss_function, optimizer_5, scheduler_5, max_epochs, dataloaders, dataset_sizes, device)


In [ ]:
import cv2
import torch
from torchvision import models
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.cuda.amp import GradScaler, autocast

# ? ?? (Haar Cascade Classifier ?)
def detect_faces_haar(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)  # RGB -> Grayscale???n    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
    return faces

# InceptionV3 ? ? ?? ?
def inception_train_with_face_detection_haar(model, model_name, loss_function, optimizer, scheduler, max_epochs, dataloaders, dataset_sizes, device):
    best_model_wts = model.state_dict()
    best_acc = 0.0
    for epoch in range(max_epochs):
        print(f'Epoch {epoch + 1}/{max_epochs}')
        print('-' * 10)

        # ?epoch ???? ?
        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            running_loss = 0.0
            running_corrects = 0

            # ???
            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)

                #  ??n                optimizer.zero_grad()

                with autocast():  # Mixed precision training ???n                    # ? ?? ?
                    for i in range(inputs.size(0)):  #  ? ?n                        img = inputs[i].cpu().numpy().transpose(1, 2, 0)  # (C, H, W) -> (H, W, C)
                        img = (img * 255).astype('uint8')  # ??0-255 ????OpenCV?  ????n                        faces = detect_faces_haar(img)
                        # ? ????? ??????? (?? ? ??????)
                        # ??? ???? ? ??.

                    # ??? ?
                    outputs = model(inputs)
                    logits = outputs[0] if isinstance(outputs, tuple) else outputs  # outputs ???? [0]???
                    _, preds = torch.max(logits, 1)  # logits???? ??
                    loss = loss_function(logits, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # ?????  ???n            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = model.state_dict()

        # ???? ??
        scheduler.step(epoch_loss)

    print(f'Best val Acc: {best_acc:.4f}')
    model.load_state_dict(best_model_wts)
    return model


# model_7 ? (InceptionV3 )
model_7 = models.inception_v3(pretrained=True)
model_7.AuxLogits.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model_7.AuxLogits.fc.in_features, n_features)
)
model_7.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model_7.fc.in_features, n_features)
)
model_7.to(device)

# ???? transform ?
inception_transform = transforms.Compose([
    transforms.Resize((299, 299)),  # InceptionV3??? ?
    transforms.ToTensor(),          # ????n    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ???n])

# ?? ?
inception_train_dataset = OpenCVDataset(train_data_dir, transform=inception_transform)
inception_val_dataset = OpenCVDataset(val_data_dir, transform=inception_transform)
inception_test_dataset = OpenCVDataset(test_data_dir, transform=inception_transform)

# ?????
batch_size = 16
inception_dataloaders = {
    'train': torch.utils.data.DataLoader(inception_train_dataset, batch_size=batch_size, shuffle=True, num_workers=8, pin_memory=True),
    'val': torch.utils.data.DataLoader(inception_val_dataset, batch_size=batch_size, shuffle=False, num_workers=8, pin_memory=True),
    'test': torch.utils.data.DataLoader(inception_test_dataset, batch_size=batch_size, shuffle=False, num_workers=8, pin_memory=True)
}

# ??? ?? ?
optimizer_7 = torch.optim.Adam(model_7.parameters(), lr=1e-5)
scheduler_7 = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_7, mode='min', factor=0.1, patience=10, verbose=True)

#  ?
inception_train_with_face_detection_haar(model_7, 'inception_v3_haar', loss_function, optimizer_7, scheduler_7, max_epochs, inception_dataloaders, dataset_sizes, device)


In [ ]:
import matplotlib.pyplot as plt
import torch

plt.figure(figsize=(12, 8))

# ResNet50
plt.plot(range(1, len(resnet50_acc_dict['val']) + 1), resnet50_acc_dict['val'], label='ResNet50')

# ResNet50 with Haar
plt.plot(range(1, len(resnet50_haar_acc_dict['val']) + 1), resnet50_haar_acc_dict['val'], label='ResNet50 (Haar)')

plt.xlabel('Epochs')
plt.ylabel('Validation Accuracy')
plt.legend()
plt.title('Validation Accuracy of Models')
plt.savefig('validation_accuracy.png')
plt.show()

In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np

# ???? ??
model_7.eval()

# ??????? ????
selected_images = []
for class_name in class_names:
    class_dir = os.path.join(test_data_dir, class_name)
    images = [os.path.join(class_dir, img) for img in os.listdir(class_dir) if img.endswith(('.png', '.jpg', '.jpeg'))]
    selected_images.append(random.choice(images))

# 2x5 ?? ?
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

for idx, img_path in enumerate(selected_images):
    # ??  ???n# External data loading omitted for portfolio archive.
    img_tensor = data_transforms['test'](img).unsqueeze(0).to(device)

    #  ?
    with torch.no_grad():
        outputs = model_7(img_tensor)
        probabilities = torch.softmax(outputs, dim=1)
        predicted_class = torch.argmax(probabilities, dim=1).item()
        confidence = probabilities[0, predicted_class].item()

    # ???n    img_np = np.array(img)
    axes[idx].imshow(img_np)
    axes[idx].axis('off')
    axes[idx].set_title(f"{class_names[predicted_class]}\n{confidence:.2f}")

# ?? 
plt.tight_layout()
plt.show()


####**model_1~5 Evaluation**

In [ ]:
model_best=model_8

In [ ]:
# best model : resnet152 + CELoss + RMSprop
## best model ? code

from torch.cuda.amp import GradScaler, autocast

scaler = GradScaler()

def train_model(model, model_name, loss_function, optimizer, scheduler, max_epochs):
    file_name = f"{model_name}_model_params.pt"
    best_acc = 0.0
    loss_dict = {"train": [], "val": []}
    acc_dict = {"train": [], "val": []}

    for epoch in range(max_epochs):
        print(f'Epoch {epoch+1}/{max_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    with autocast():
                        outputs = model(inputs)
                        _, preds = torch.max(outputs, 1)
                        loss = loss_function(outputs, labels)

                    if phase == 'train':
                        scaler.scale(loss).backward()
                        scaler.step(optimizer)
                        scaler.update()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            loss_dict[phase].append(epoch_loss)
            acc_dict[phase].append(epoch_acc.item())

            plt.plot(range(len(loss_dict[phase])), loss_dict[phase])
            plt.savefig(f"{phase}_Loss.png")
            plt.close()

            plt.plot(range(len(acc_dict[phase])), acc_dict[phase])
            plt.savefig(f"{phase}_ACC.png")
            plt.close()

            print(f'{phase} Total Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                torch.save(model.state_dict(), file_name)

            if phase == 'val':
                scheduler.step(epoch_loss)

        print()

    print(f'Best val Acc: {best_acc:.4f}')
    torch.save(loss_dict, f"{model_name}_loss_dict.pth")
    torch.save(acc_dict, f"{model_name}_acc_dict.pth")

n_features = 6
model_best = models.resnet152(pretrained=True)
model_best.fc = nn.Sequential(
    nn.Dropout(0.5),  # Dropout
    nn.Linear(model_best.fc.in_features, n_features)  # ??fully connected layer
)
model_best.to(device)
optimizer = torch.optim.RMSprop(model_best.parameters(), lr=1e-5) # optimizer ?
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=10, verbose=True)

loss_function_best= nn.CrossEntropyLoss()

train_model(model_best, 'RMSprop', loss_function_best, optimizer, scheduler, max_epochs)

### **4. ???model?test ??  ?6?? (? 10?**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
import torch.nn.functional as F

def visualize_random_predictions(model, dataloader, class_names, num_images=10):
    model.eval()
    all_inputs = []
    all_labels = []

    for inputs, labels in dataloader:
        for i in range(inputs.size()[0]):
            all_inputs.append(inputs[i])
            all_labels.append(labels[i])

    #?? ?? ?
    random_indices = random.sample(range(len(all_inputs)), num_images)
    selected_images = [all_inputs[idx] for idx in random_indices]
    selected_labels = [all_labels[idx] for idx in random_indices]

    with torch.no_grad():
        for i in range(num_images):
            inputs = selected_images[i].unsqueeze(0).to(device)
            labels = selected_labels[i].to(device)

            outputs = model(inputs)
            probs = F.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)

            fig, ax = plt.subplots(1, 2, figsize=(5, 5))
            ax[0].axis('off')
            img = inputs.cpu().squeeze(0).permute(1, 2, 0).numpy()
            img = np.clip(img, 0, 1)
            ax[0].imshow(img)
            ax[0].set_title(f'Predicted: {class_names[preds.item()]}')

            prob_text = "\n".join([f"{class_names[j]}: {probs[0][j]:.4f}" for j in range(len(class_names))])
            ax[1].text(0.1, 0.5, prob_text, fontsize=12, verticalalignment='center')
            ax[1].axis('off')

            plt.tight_layout()
            plt.show()

n_features = 6
model_best = models.resnet152(pretrained=True)
model_best.fc = nn.Sequential(
    nn.Dropout(0.5),  # Dropout
    nn.Linear(model_best.fc.in_features, n_features)  # ??fully connected layer
)
model_best.to(device)

model_best.load_state_dict(torch.load('RMSprop_model_params.pt'))

visualize_random_predictions(model_best, dataloaders['test'], class_names)  #test ?? ?? ????


### **5. ???model??? ?? **

In [ ]:
def load_image(image_data, transform=None):
# External data loading omitted for portfolio archive.
    if transform:
        image = transform(image).unsqueeze(0)   # ?
    return image

def predict_image(model, image_data, transform, class_names):
    image = load_image(image_data, transform)
    image = image.to(device)

    with torch.no_grad():
        outputs = model(image)
        probs = torch.nn.functional.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)

    return preds.item(), probs.cpu().numpy()

def visualize_prediction(image_data, pred_class, pred_probs, class_names):
# External data loading omitted for portfolio archive.
    plt.imshow(image)
    plt.title(f'Predicted: {class_names[pred_class]}')
    plt.axis('off')
    plt.show()

    print("Prediction Probabilities:")
    for i, class_name in enumerate(class_names):
        print(f"{class_name}: {pred_probs[0][i]:.4f}")

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
import io
from IPython.display import display


n_features = 6
model_best = models.resnet152(pretrained=True)
model_best.fc = nn.Sequential(
    nn.Dropout(0.5),  # Dropout
    nn.Linear(model_best.fc.in_features, n_features)  # ??fully connected layer
)
model_best.to(device)

model_best.load_state_dict(torch.load('RMSprop_model_params.pt'))

    image_data = next(iter(change['new'].values()))['content']
    pred_class, pred_probs = predict_image(model_best, image_data, transform, class_names)
    visualize_prediction(image_data, pred_class, pred_probs, class_names)




In [ ]:
import io
from IPython.display import display

n_features = 6
model_best = models.resnet152(pretrained=True)
model_best.fc = nn.Sequential(
    nn.Dropout(0.5),  # Dropout
    nn.Linear(model_best.fc.in_features, n_features)  # ??fully connected layer
)
model_best.to(device)

model_best.load_state_dict(torch.load('RMSprop_model_params.pt'))

    image_data = next(iter(change['new'].values()))['content']
    pred_class, pred_probs = predict_image(model_best, image_data, transform, class_names)
    visualize_prediction(image_data, pred_class, pred_probs, class_names)




In [ ]:
import io
from IPython.display import display

n_features = 6
model_best = models.resnet152(pretrained=True)
model_best.fc = nn.Sequential(
    nn.Dropout(0.5),  # Dropout
    nn.Linear(model_best.fc.in_features, n_features)  # ??fully connected layer
)
model_best.to(device)

model_best.load_state_dict(torch.load('RMSprop_model_params.pt'))

    image_data = next(iter(change['new'].values()))['content']
    pred_class, pred_probs = predict_image(model_best, image_data, transform, class_names)
    visualize_prediction(image_data, pred_class, pred_probs, class_names)




In [ ]:
import io
from IPython.display import display

n_features = 6
model_best = models.resnet152(pretrained=True)
model_best.fc = nn.Sequential(
    nn.Dropout(0.5),  # Dropout
    nn.Linear(model_best.fc.in_features, n_features)  # ??fully connected layer
)
model_best.to(device)

model_best.load_state_dict(torch.load('RMSprop_model_params.pt'))

    image_data = next(iter(change['new'].values()))['content']
    pred_class, pred_probs = predict_image(model_best, image_data, transform, class_names)
    visualize_prediction(image_data, pred_class, pred_probs, class_names)


